# Sensitivity and specificity analysis: figures

Clean version of `sensitivity_and_specificity_analysis.ipynb` with only the cells that make the final figures:

* box plots of sensitivity and specificity per marked lineage, each embryo coloured by its Procrustes distance;
* sensitivity and specificity against embryo age, both marked lineages in one figure per rate;
* sensitivity and specificity against embryo age, one figure per marked lineage (PDF 120 mm × 44 mm);
* mean and median sensitivity and specificity after excluding embryos above a Procrustes distance threshold (table).

Lineage assignment: `Assigned_name`, the cell names of the pattern selected by the modified Procrustes distance;
distance: `min_Procrustes_mod_distance`. Sensitivity = TP/(TP + FN) and specificity = TN/(TN + FP), counted per embryo
against the lineage-tracing ground truth.

Run order: 04_a (with `df_reduced2.pkl`) and `unlabeled_data_alignment_P_cell_divided.py` → `combined_experiments.ipynb`
(optional; only for the mean self-confidence in the hover text) → this notebook.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from matplotlib import cbook
from plotly.subplots import make_subplots
from scipy import stats

import config

pio.kaleido.scope.mathjax = None   # stops kaleido printing a "Loading [MathJax]..." box in the PDFs

LINEAGES = ['E', 'ABa']

# CeSCALE names and the per-embryo Procrustes distance of the alignment method.
LABEL_COLUMNS = {'Procrustes mod': 'Assigned_name'}
DIST_COLUMNS = {'Procrustes mod': 'min_Procrustes_mod_distance'}

# What to do with embryos whose alignment results do not match their segmentation file: 'raise' (stop) or 'skip'.
ON_MISMATCH = 'raise'
XY_TOLERANCE = 1e-3   # px

# Set to True to write the figures to FIG_DIR (plotly: HTML + PDF, matplotlib: PDF + PNG).
SAVE_FIGURES = True
FIG_DIR = Path('figures') / 'sensitivity_and_specificity_analysis'

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 12,
    'axes.linewidth': 1.5,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'pdf.fonttype': 42,
})

## Inputs

`ALIGNMENT_RUNS` lists each CeSCALE run with the segmentation file it was computed from. `DATASETS` has one row per
ground-truth file:

* `pattern` - regular expression for the whole `sample_id` of the embryos of the dataset; the group `image` is the
  image name used in the ground-truth files;
* `layout` - how the segmentation file stores the P-cells: `single` (one P-cell, P4; scores the images with one P-cell
  label) or `p_cells_last` (Z2 and Z3 after the other cells; scores all images).

In [ ]:
SEG_DIR = Path(config.SAVE_DIR) / 'analysis_files' / 'segmentation_results'
INPUT_DIR = Path('experiments_related_files')
GT_EARLY_DIR = INPUT_DIR / '261101_ground_truth_df_reduced2_rerun'

ALIGNMENT_RUNS = {
    'P-cell divided': dict(
        alignment=SEG_DIR / 'alignment_results_Nr' / 'df_aligned_XYZ_complete_Nr.csv',
        segmentation=SEG_DIR / 'df_reduced3.pkl'),
    'P-cell not divided': dict(
        alignment=SEG_DIR / 'alignment_results' / 'df_aligned_XYZ_complete.csv',
        segmentation=SEG_DIR / 'df_reduced2.pkl'),
}

DATASETS = pd.DataFrame([
    ('E', 1, 'P-cell divided', r'R1_merged(?P<image>405_0_\d+_FOV\d+)',
     INPUT_DIR / 'Lineage_Numbers_LATE_R1.csv', INPUT_DIR / 'P-cell_Numbers_LATE_R1_.csv', 'p_cells_last'),
    ('E', 2, 'P-cell divided', r'R2_merged(?P<image>405_0_\d+_FOV\d+)',
     INPUT_DIR / 'Lineage_Numbers_LATE_R2.csv', INPUT_DIR / 'P-cell_Numbers_LATE_R2.csv', 'p_cells_last'),
    ('E', 3, 'P-cell divided', r'R3_merged(?P<image>405_0_\d+_FOV\d+)',
     INPUT_DIR / 'Lineage_Numbers_LATE_R3.csv', INPUT_DIR / 'P-cell_Numbers_LATE_R3.csv', 'p_cells_last'),
    ('ABa', 2, 'P-cell not divided', r'early_R2_(?P<image>DAPI_0_\d+_FOV\d+)',
     GT_EARLY_DIR / 'Lineage Numbers_R2.csv', GT_EARLY_DIR / 'P-cell Numbers_R2.csv', 'single'),
    ('ABa', 3, 'P-cell not divided', r'early_R3_(?P<image>DAPI_0_\d+_FOV\d+)',
     GT_EARLY_DIR / 'Lineage Numbers_R3.csv', GT_EARLY_DIR / 'P-cell Numbers_R3.csv', 'single'),
], columns=['Lineage', 'Replicate', 'run', 'pattern', 'lineage_csv', 'pcell_csv', 'layout'])

# Per-cell self-confidence (combined_experiments.ipynb), only for the hover text. None or missing: skipped. Scores of
# an embryo are used only if they were computed on the same cells as the alignment results scored here.
CONFIDENCE_FILE = SEG_DIR / 'combined_experiments' / 'combined_per_cell_confidence_all_embryos.csv'
CONF_COL = 'Relative_self_confidence(Procrustes_mod)'

DATASETS[['Lineage', 'Replicate', 'run', 'layout']].assign(
    lineage_csv=DATASETS['lineage_csv'].map(lambda p: Path(p).name))

## Load the alignment results and check them against the segmentation

Re-segmentation renumbers the labels, so alignment results of an older segmentation would be scored against the wrong
cells without any error. For every embryo, the cells of the alignment results (`cell_name`, `Initial_X`, `Initial_Y`)
must equal those of the segmentation file, and the P-cell must be where the layout expects it. By default the notebook
stops on a mismatch (`ON_MISMATCH`).

In [ ]:
def read_label_csv(path):
    # Rows '<image>_cp_masks...,"12,13"' after a header line -> {'<image>': [12, 13]}.
    label_map = {}
    for line in Path(path).read_text().splitlines()[1:]:
        if ',' not in line:
            continue
        name, text = line.split(',', 1)
        name = re.sub(r'_cp_masks.*', '', name.strip())
        text = text.strip().replace('"', '')
        if not name or not text:
            continue
        ids = pd.to_numeric(pd.Series(text.split(',')).str.strip(), errors='coerce').dropna()
        label_map[name] = [int(i) for i in ids]
    return label_map


def match_image(sample_id, pattern):
    m = re.fullmatch(pattern, str(sample_id))
    return m.group('image') if m else None


alignment, segmentation = {}, {}
for run, files in ALIGNMENT_RUNS.items():
    path = files['alignment']
    if path is None or not Path(path).exists():
        print(f'{run}: no alignment results ({path}); its embryos are not scored.')
    else:
        alignment[run] = pd.read_csv(path, low_memory=False, dtype={'sample_id': str})
        print(f'{run}: {alignment[run]["sample_id"].nunique()} sample_ids (all embryos of the run) in {path}')
    segmentation[run] = [d.reset_index(drop=True) for d in pd.read_pickle(files['segmentation'])]

# EMBRYOS[dataset index] = {image: (sample_id, cells of the embryo in cell_name order)}
EMBRYOS, SEGMENTED = {}, {}
for ds in DATASETS.itertuples():
    embryos = {}
    cells = alignment.get(ds.run)
    if cells is not None:
        for sample_id, emb in cells.groupby('sample_id', sort=False):
            image = match_image(sample_id, ds.pattern)
            if image is None:
                continue
            if image in embryos:
                raise ValueError(f'{ds.run}: two embryos for {image} ({embryos[image][0]}, {sample_id})')
            embryos[image] = (sample_id, emb.sort_values('cell_name').reset_index(drop=True))
    EMBRYOS[ds.Index] = embryos

    segmented = {}
    for d in segmentation[ds.run]:
        image = match_image(d['sample_id'].iloc[0], ds.pattern)
        if image is not None:
            if image in segmented:
                raise ValueError(f'{ALIGNMENT_RUNS[ds.run]["segmentation"]}: two embryos for {image}')
            segmented[image] = d.sort_values('cell_name').reset_index(drop=True)
    SEGMENTED[ds.Index] = segmented

In [ ]:
# Alignment results vs segmentation file, embryo by embryo.
check_rows = []
for ds in DATASETS.itertuples():
    pcells_gt = read_label_csv(ds.pcell_csv)
    for image, (sample_id, emb) in EMBRYOS[ds.Index].items():
        seg = SEGMENTED[ds.Index].get(image)
        n = len(emb)
        row = dict(Lineage=ds.Lineage, Replicate=ds.Replicate, run=ds.run, ImageName=image, sample_id=sample_id,
                   n_alignment=n, n_segmentation=np.nan if seg is None else len(seg), max_xy_difference=np.nan)
        problems = []
        if (emb['cell_name'].to_numpy() != np.arange(1, n + 1)).any():
            problems.append('cell_name is not 1..n')
        if seg is None:
            problems.append('not in the segmentation file')
        elif len(seg) != n:
            problems.append('different number of cells')
        else:
            diff = np.abs(emb[['Initial_X', 'Initial_Y']].to_numpy(float) - seg[['X', 'Y']].to_numpy(float)).max()
            row['max_xy_difference'] = diff
            if not diff <= XY_TOLERANCE:
                problems.append(f'cell positions differ (max {diff:.3g} px)')
        pcell_id = int(emb['Pcell_id'].iloc[0])
        if ds.layout == 'single':
            gt = pcells_gt.get(image)
            if gt is not None and len(gt) == 1 and gt[0] != pcell_id:
                problems.append(f'Pcell_id {pcell_id}, P-cell file {gt[0]}')
        elif pcell_id != n:
            problems.append(f'Pcell_id {pcell_id} is not the last cell ({n})')
        row['problems'] = '; '.join(problems)
        check_rows.append(row)

segmentation_check = pd.DataFrame(check_rows, columns=[
    'Lineage', 'Replicate', 'run', 'ImageName', 'sample_id', 'n_alignment', 'n_segmentation', 'max_xy_difference',
    'problems'])
# An embryo can be checked by several dataset rows (one per ground-truth file): one row per embryo, all problems.
segmentation_check = (segmentation_check
                      .groupby(['run', 'sample_id'], sort=False, as_index=False)
                      .agg({**{c: 'first' for c in segmentation_check.columns if c not in ('run', 'sample_id')},
                            'problems': lambda p: '; '.join(dict.fromkeys(x for x in p if x))}))
display(segmentation_check.assign(ok=segmentation_check['problems'] == '')
        .groupby(['run', 'Lineage', 'Replicate'])['ok'].agg(embryos='size', match='sum'))

# Embryos of the segmentation files without alignment results (e.g. fewer than 3 cells, or the run is missing).
for run in ALIGNMENT_RUNS:
    images = {i for ds in DATASETS[DATASETS['run'] == run].itertuples() for i in SEGMENTED[ds.Index]}
    aligned = {i for ds in DATASETS[DATASETS['run'] == run].itertuples() for i in EMBRYOS[ds.Index]}
    if images - aligned:
        print(f'{run}: no alignment results for {", ".join(sorted(images - aligned))}')

mismatch = segmentation_check[segmentation_check['problems'] != '']
SKIPPED = set(zip(mismatch['run'], mismatch['sample_id']))   # (alignment run, sample_id) left out of the counts
if len(mismatch):
    display(mismatch)
    message = (f'{len(mismatch)} embryo(s) whose alignment results do not match their segmentation file. They were '
               'aligned on another segmentation.')
    if ON_MISMATCH == 'raise':
        raise ValueError(message + ' Rerun the alignment, or set ON_MISMATCH = "skip" to leave them out.')
    print(message + ' They are left out (ON_MISMATCH = "skip").')

## Confusion counts

* **predicted positive** - a cell whose CeSCALE name starts with the marked lineage (`E`, `ABa`);
* **ground truth** - the segmentation label ids of the marked cells per image (`Lineage*Numbers*.csv`);
* CeSCALE cell *i* (`cell_name`) is mapped to a segmentation label id according to the `layout`: `single`, cell *i* is
  label *i*; `p_cells_last`, the CeSCALE cells map to the non-P-cell labels of 1..N in order, followed by the P-cell
  labels of the P-cell file, where N is the number of segmentation labels: the number of CeSCALE cells, plus one when
  Z2 and Z3 (2 P-cell labels) were merged into one cell before CeSCALE;
* TP, FP and FN are counted over label ids; TN covers the remaining labels of the embryo, P-cells included;
* **embryo age** is the number of segmented cells, TP + FP + FN + TN.

Every ground-truth image is either scored once or listed with the reason it is not.

In [ ]:
COUNT_COLS = ['TP', 'FP', 'FN', 'TN']
# Number of P-cell labels of the images a dataset row scores (None: any).
P_CELL_LABELS = {'single': {1}, 'p_cells_last': None}


def label_map(layout, n_labels, pcell_ids):
    # CeSCALE cell i (cell_name, 1-based) -> segmentation label id: label_ids[i - 1]; and all label ids of the embryo.
    # n_labels: number of segmentation labels of the embryo.
    if layout == 'single':
        label_ids = list(range(1, n_labels + 1))
    elif layout == 'p_cells_last':
        label_ids = [l for l in range(1, n_labels + 1) if l not in pcell_ids] + list(pcell_ids)
    else:
        raise ValueError(f'unknown layout {layout}')
    return label_ids, set(label_ids)


def classify_dataset(ds, label_col, verbose=True):
    # Confusion counts of the ground-truth images of one dataset row; status of every image the row is responsible for.
    marked = read_label_csv(ds.lineage_csv)
    p_cells = read_label_csv(ds.pcell_csv)
    allowed = P_CELL_LABELS[ds.layout]
    rows, status, no_pcells = [], {}, []
    for image in sorted(marked):
        pcell_ids = p_cells.get(image, [])
        if allowed is not None and len(pcell_ids) not in allowed:
            continue   # an image of another layout
        if image not in EMBRYOS[ds.Index]:
            status[image] = 'no alignment results'
            continue
        sample_id, emb = EMBRYOS[ds.Index][image]
        if (ds.run, sample_id) in SKIPPED:
            status[image] = 'left out (segmentation check)'
            continue
        if image not in p_cells:
            no_pcells.append(image)
        n = len(emb)
        # Z2 and Z3 (2 P-cell labels) were merged into one cell before CeSCALE: one label more than CeSCALE cells.
        n_labels = n + 1 if ds.layout == 'p_cells_last' and len(pcell_ids) == 2 else n
        label_ids, all_labels = label_map(ds.layout, n_labels, pcell_ids)
        if len(label_ids) != n_labels:
            status[image] = f'{len(label_ids)} label ids for {n_labels} labels ({n} CeSCALE cells)'
            continue
        true = set(marked[image])
        outside = sorted(true - all_labels)
        if outside and verbose:
            print(f'WARNING {ds.Lineage} R{ds.Replicate} {image}: ground-truth labels {outside} are not labels of the '
                  f'embryo (1..{max(all_labels)})')
        predicted = {label_ids[c - 1] for c, name in zip(emb['cell_name'], emb[label_col].astype(str))
                     if name.startswith(ds.Lineage)}
        rows.append([ds.Lineage, ds.Replicate, image, sample_id, ds.layout, len(predicted & true),
                     len(predicted - true), len(true - predicted), len(all_labels - (predicted | true))])
        status[image] = 'scored'
    if verbose and no_pcells:
        print(f'{ds.Lineage} R{ds.Replicate}: not in {Path(ds.pcell_csv).name}, assuming no P-cells: '
              f'{", ".join(no_pcells)}')

    df = pd.DataFrame(rows, columns=['Lineage', 'Replicate', 'ImageName', 'sample_id', 'layout', *COUNT_COLS])
    df['Sensitivity'] = df['TP'] / (df['TP'] + df['FN'])
    df['Specificity'] = df['TN'] / (df['TN'] + df['FP'])
    df['Age'] = df[COUNT_COLS].sum(axis=1)
    return df, status


results, coverage_rows = {}, []
for i, (method, label_col) in enumerate(LABEL_COLUMNS.items()):
    frames = []
    for ds in DATASETS.itertuples():
        df, status = classify_dataset(ds, label_col, verbose=(i == 0))
        frames.append(df)
        if i == 0:
            coverage_rows += [dict(Lineage=ds.Lineage, Replicate=ds.Replicate, ground_truth=Path(ds.lineage_csv).name,
                                   layout=ds.layout, ImageName=image, status=s) for image, s in status.items()]
    results[method] = pd.concat([f for f in frames if len(f)] or frames[:1], ignore_index=True)
    duplicated = results[method].duplicated(['Lineage', 'Replicate', 'ImageName'], keep=False)
    assert not duplicated.any(), results[method][duplicated]

# Ground-truth images that no dataset row is responsible for (P-cell count fits no layout).
coverage = pd.DataFrame(coverage_rows)
for (lineage_csv, pcell_csv), part in DATASETS.groupby(['lineage_csv', 'pcell_csv'], sort=False):
    claimed = set(coverage.loc[coverage['ground_truth'] == Path(lineage_csv).name, 'ImageName'])
    p_cells = read_label_csv(pcell_csv)
    for image in sorted(set(read_label_csv(lineage_csv)) - claimed):
        coverage.loc[len(coverage)] = dict(Lineage=part['Lineage'].iloc[0], Replicate=part['Replicate'].iloc[0],
                                           ground_truth=Path(lineage_csv).name, layout='-', ImageName=image,
                                           status=f'{len(p_cells.get(image, []))} P-cell labels, no dataset row')

display(coverage.pivot_table(index=['Lineage', 'Replicate'], columns='status', values='ImageName', aggfunc='count',
                             fill_value=0))
not_scored = coverage[coverage['status'] != 'scored']
if len(not_scored):
    print('Ground-truth images not scored:')
    display(not_scored.sort_values(['Lineage', 'Replicate', 'ImageName']).reset_index(drop=True))
for method, df in results.items():
    for lineage in LINEAGES:
        if not (df['Lineage'] == lineage).any():
            print(f'WARNING {method}: no scored {lineage} embryos (see the table above); the figures have no '
                  f'{lineage} data.')

## Plotting helpers

Box plots follow the MATLAB `boxplot(..., 'Notch', 'on')` convention: the box spans the quartiles, whiskers reach the
most extreme points within 1.5 IQR, and the notch is the 95 % confidence interval of the median,
median ± 1.57 IQR / √n. When that interval extends past a quartile, the notch shows up as a flap outside the box.

The dotted lines around a linear fit are the 95 % confidence bounds of the fitted line (not prediction bounds for
single embryos).

The HTML figures show the details of each embryo on hover: counts, Procrustes distance, and the mean self-confidence
when `CONFIDENCE_FILE` has up-to-date scores for the embryo.

In [ ]:
Y_LIM = (0, 1.1)
Y_TICKS = np.arange(0, 1.01, 0.2)
AGE_LIM = (0, 250)
BIG_FONT = 18     # font size of the figures (px)
BIG_MARKER = 13   # marker size of the figures (px)
PDF_WIDTH_MM, PDF_HEIGHT_MM = 120, 44   # PDF page of the per-lineage figures
PX_PER_MM = 96 / 25.4   # kaleido writes PDFs at 96 px per inch (1 px = 0.75 pt)


def linear_fit_with_ci(x, y, level=0.95, n_grid=200):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    fit = stats.linregress(x, y)
    residual_sd = np.sqrt(np.sum((y - (fit.intercept + fit.slope * x)) ** 2) / (n - 2))
    t_crit = stats.t.ppf(0.5 + level / 2, n - 2)
    fit.slope_ci = (fit.slope - t_crit * fit.stderr, fit.slope + t_crit * fit.stderr)   # 95 % CI of the slope
    x_grid = np.linspace(x.min(), x.max(), n_grid)
    y_grid = fit.intercept + fit.slope * x_grid
    half_width = t_crit * residual_sd * np.sqrt(1 / n + (x_grid - x.mean()) ** 2 / np.sum((x - x.mean()) ** 2))
    return fit, x_grid, y_grid, y_grid - half_width, y_grid + half_width


def fit_table(fits):
    return pd.DataFrame({
                rate: {'slope (per cell)': f.slope, 'slope 95% CI low': f.slope_ci[0], 'slope 95% CI high': f.slope_ci[1],
               'intercept': f.intercept, 'r^2': f.rvalue ** 2, 'p-value': f.pvalue}
# rate: {'slope (per cell)': f.slope, 'intercept': f.intercept, 'r^2': f.rvalue ** 2, 'p-value': f.pvalue}
        for rate, f in fits.items()
    }).T


def save_figure(fig, name):
    # matplotlib figure: PDF + PNG.
    if not SAVE_FIGURES:
        return
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    stem = 'sensitivity_and_specificity_' + name.replace(' ', '_').replace(',', '')
    for ext in ['pdf', 'png']:
        fig.savefig(FIG_DIR / f'{stem}.{ext}', dpi=300, bbox_inches='tight')


def save_figure_plotly(fig, name, width_mm=None):
    # plotly figure: HTML (interactive) + PDF. width_mm: PDF page width in mm, the figure scaled as a whole (fonts,
    # markers and lines included) and the height following the layout's aspect ratio; None: the layout size.
    if not SAVE_FIGURES:
        return
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    stem = FIG_DIR / ('sensitivity_and_specificity_' + name.replace(' ', '_').replace(',', ''))
    fig.write_html(f'{stem}_plotly.html')
    fig.write_image(f'{stem}.pdf', scale=1 if width_mm is None else width_mm * PX_PER_MM / fig.layout.width)


# Per-embryo details for the hover text and the distance colouring, one row per sample_id.
EMBRYO_COLS = ['min_Procrustes_mod_distance', 'Ambiguous_state']
CURRENT_EMBRYOS = {sid: emb for embryos in EMBRYOS.values() for sid, emb in embryos.values()}
EMBRYO_DETAILS = pd.DataFrame.from_dict(
    {sid: emb.iloc[0].reindex(EMBRYO_COLS) for sid, emb in CURRENT_EMBRYOS.items()}, orient='index')


def load_confidence():
    # Mean self-confidence per embryo, for the embryos whose scores were computed on the cells scored here.
    if CONFIDENCE_FILE is None or not Path(CONFIDENCE_FILE).exists():
        print(f'No confidence file ({CONFIDENCE_FILE}); the hover text has no self-confidence.')
        return None
    conf = pd.read_csv(CONFIDENCE_FILE, usecols=['sample_id', 'cell_name', 'Initial_X', 'Initial_Y', CONF_COL, 'Pm'],
                       dtype={'sample_id': str}, low_memory=False)
    rows, stale = {}, []
    for sample_id, part in conf.groupby('sample_id'):
        emb = CURRENT_EMBRYOS.get(sample_id)
        if emb is None:
            continue
        part = part.sort_values('cell_name')
        same = (len(part) == len(emb) and (part['cell_name'].to_numpy() == emb['cell_name'].to_numpy()).all()
                and np.allclose(part[['Initial_X', 'Initial_Y']].to_numpy(float),
                                emb[['Initial_X', 'Initial_Y']].to_numpy(float), atol=XY_TOLERANCE))
        if not same:
            stale.append(sample_id)
            continue
        rows[sample_id] = {'mean_confidence': part[CONF_COL].mean(), 'ref_images': part['Pm'].iloc[0]}
    missing = sorted(set(CURRENT_EMBRYOS) - set(rows) - set(stale))
    print(f'Self-confidence of {len(rows)} of {len(CURRENT_EMBRYOS)} embryos from {Path(CONFIDENCE_FILE).name}.')
    if stale:
        print(f'  {len(stale)} computed on other cells (out of date, rerun 06_a/06_b and combined_experiments), '
              f'not used: {", ".join(stale)}')
    if missing:
        print(f'  {len(missing)} not in the file: {", ".join(missing)}')
    return pd.DataFrame.from_dict(rows, orient='index') if rows else None


CONFIDENCE = load_confidence()


def add_embryo_details(df):
    df = df.join(EMBRYO_DETAILS, on='sample_id')
    if CONFIDENCE is not None:
        df = df.join(CONFIDENCE, on='sample_id')
    return df


COLUMN_LABELS = {'min_Procrustes_mod_distance': 'Procrustes distance'}
# Sequential blue ramp: light = small distance (good fit), dark = large distance. The light end stays visible on white.
SEQUENTIAL_BLUE = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#0d366b']
LINEAGE_COLORS = {'E': '#2a78d6', 'ABa': '#eb6834'}


def hover_texts(df):
    def num(v, spec='.4f'):
        return 'n/a' if pd.isna(v) else format(v, spec)

    texts = []
    for r in df.itertuples():
        text = (f'<b>{r.ImageName}</b>  ({r.sample_id})<br>'
                f'marked lineage: {r.Lineage}, replicate R{r.Replicate}<br>'
                f'age: {r.Age} cells, ambiguous state: {r.Ambiguous_state}<br>'
                f'TP {r.TP}, FP {r.FP}, FN {r.FN}, TN {r.TN}<br>'
                f'sensitivity: {r.Sensitivity:.3f}, specificity: {r.Specificity:.3f}<br>'
                f'Procrustes distance: {num(r.min_Procrustes_mod_distance)}')
        if 'mean_confidence' in df:
            text += f'<br>mean self-confidence: {num(r.mean_confidence, ".3f")}'
            if pd.notna(r.ref_images):
                text += f' ({int(r.ref_images)} reference images for the pattern'
                text += ', excluded from the confidence figures)' if r.ref_images <= 1 else ')'
        texts.append(text)
    return texts


def style_axes(fig):
    fig.update_xaxes(showline=True, mirror=True, linewidth=1.5, linecolor='black', showgrid=False, zeroline=False)
    fig.update_yaxes(showline=True, mirror=True, linewidth=1.5, linecolor='black', showgrid=False, zeroline=False,
                     range=list(Y_LIM), tickvals=Y_TICKS, ticks='inside')


def add_rate_boxes(fig, df, rate, col, rng, color_by=None, lineages=LINEAGES):
    for x, lineage in enumerate(lineages, start=1):
        part = df[df['Lineage'] == lineage]
        if part.empty:
            fig.add_annotation(x=x, y=0.5, text='<i>no data</i>', showarrow=False, font=dict(color='grey'),
                               row=1, col=col)
            continue
        # Whiskers within 1.5 IQR, notch = 95 % CI of the median.
        s = cbook.boxplot_stats(part[rate].to_numpy(dtype=float))[0]
        fig.add_trace(go.Box(
            x=[x], q1=[s['q1']], median=[s['med']], q3=[s['q3']],
            lowerfence=[s['whislo']], upperfence=[s['whishi']],
            notched=True, notchspan=[(s['cihi'] - s['cilo']) / 2], width=0.5,
            fillcolor='rgba(0,0,0,0)', line=dict(color='black', width=1.5),
            name=lineage, showlegend=False,
        ), row=1, col=col)

        outlier = ((part[rate] < s['whislo']) | (part[rate] > s['whishi'])).to_numpy()
        jitter = rng.uniform(-0.15, 0.15, len(part))
        if color_by is None:
            groups = [
                (~outlier, dict(symbol='circle', size=5, color='rgba(0, 0, 0, 0.35)')),
                (outlier, dict(symbol='circle-open', size=8, color='rgb(166, 166, 166)', line=dict(width=1.5))),
            ]
        else:
            # One trace coloured by `color_by`; box plot outliers get a black ring.
            groups = [(np.ones(len(part), dtype=bool), dict(
                size=9, color=part[color_by], coloraxis='coloraxis',
                line=dict(color=np.where(outlier, 'black', 'white'), width=np.where(outlier, 2, 1)),
            ))]
        texts = np.array(hover_texts(part), dtype=object)
        for mask, marker in groups:
            if mask.any():
                fig.add_trace(go.Scatter(
                    x=x + jitter[mask], y=part[rate].to_numpy()[mask], mode='markers', marker=marker,
                    hovertext=texts[mask], hovertemplate='%{hovertext}<extra></extra>', showlegend=False,
                ), row=1, col=col)


def accuracy_by_distance(details, threshold, exclude, dist_col):
    # Mean and median sensitivity / specificity after excluding embryos by their Procrustes distance.
    # details: add_embryo_details(results[method]). exclude='below': drop embryos with distance < threshold (keeps the
    # worse-aligned ones); exclude='above': drop embryos with distance > threshold (keeps the better-aligned ones).
    # Each embryo counts once, as in the box plots.
    if exclude == 'below':
        kept = details[details[dist_col] >= threshold]
    elif exclude == 'above':
        kept = details[details[dist_col] <= threshold]
    else:
        raise ValueError("exclude must be 'below' or 'above'")

    rows = {}
    for name in [*LINEAGES, 'all']:
        part = kept if name == 'all' else kept[kept['Lineage'] == name]
        total = len(details) if name == 'all' else int((details['Lineage'] == name).sum())
        rows[name] = {
            'embryos kept': len(part), 'excluded': total - len(part),
            'mean sensitivity': part['Sensitivity'].mean(), 'median sensitivity': part['Sensitivity'].median(),
            'mean specificity': part['Specificity'].mean(), 'median specificity': part['Specificity'].median(),
        }
    return pd.DataFrame.from_dict(rows, orient='index')

In [ ]:
def make_box_figure_by_distance(df, dist_col, color_range=None, lineages=LINEAGES, seed=0, panel_label='d'):
    # Box plots of sensitivity and specificity per marked lineage, each embryo coloured by its distance.
    # color_range: (min, max) of the colour scale; None: the range of the embryos shown.
    df = add_embryo_details(df[df['Lineage'].isin(lineages)])
    if color_range is None:
        color_range = (df[dist_col].min(), df[dist_col].max())
    rng = np.random.default_rng(seed)   # jitter of the points
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.16)

    for col, rate, formula in [(1, 'Sensitivity', 'TP/(TP + FN)'), (2, 'Specificity', 'TN/(TN + FP)')]:
        add_rate_boxes(fig, df, rate, col, rng, color_by=dist_col, lineages=lineages)
        axis = '' if col == 1 else str(col)
        fig.add_annotation(xref=f'x{axis} domain', yref=f'y{axis} domain', x=0.97, y=0.03, text=formula,
                           showarrow=False, xanchor='right', yanchor='bottom')
        fig.update_xaxes(tickvals=list(range(1, len(lineages) + 1)), ticktext=lineages,
                         range=[0.4, len(lineages) + 0.6], title_text='marked lineage', ticks='', row=1, col=col)
        fig.update_yaxes(title_text=rate.lower(), row=1, col=col)
    fig.update_traces(marker_size=BIG_MARKER, selector=dict(type='scatter'))
    fig.update_traces(line_width=2, selector=dict(type='box'))

    style_axes(fig)
    if panel_label:
        fig.add_annotation(xref='x domain', yref='y domain', x=-0.26, y=1.03, text=f'<b>{panel_label}</b>',
                           showarrow=False, xanchor='left', yanchor='bottom', font=dict(size=BIG_FONT + 6))
    label = COLUMN_LABELS.get(dist_col, dist_col)
    counts = ', '.join(f'{l}: n = {(df["Lineage"] == l).sum()}' for l in lineages)
    fig.update_layout(
        title=dict(text=f'({counts})<br>coloured by {label}', x=0.01, xanchor='left', y=0.955,
                   yanchor='top'),
        coloraxis=dict(colorscale=SEQUENTIAL_BLUE, cmin=color_range[0], cmax=color_range[1],
                       colorbar=dict(title=dict(text=label, side='right'), thickness=18, len=0.9)),
        template='simple_white', width=950, height=650, font=dict(family='Arial', size=BIG_FONT),
        hoverlabel=dict(align='left'), margin=dict(l=90, r=30, t=120, b=90),
    )
    return fig


def add_colored_fit(fig, x, y, color, name, show_legend, row, col, x_name='distance'):
    # Linear fit with its 95 % confidence bounds (linear_fit_with_ci), drawn in one colour.
    fit, x_grid, y_grid, lo, hi = linear_fit_with_ci(x, y)
    fit_hover = (f'{name}<br>{y.name.lower()} = {fit.intercept:.3f} + {fit.slope:.5g} × {x_name}<br>'
                 f'r² = {fit.rvalue ** 2:.3f}, p = {fit.pvalue:.3g}<extra></extra>')
    fig.add_trace(go.Scatter(
        x=x_grid, y=y_grid, mode='lines', name=f'{name}: linear fit', legendgroup=name, showlegend=show_legend,
        line=dict(color=color, width=3), hovertemplate=fit_hover,
    ), row=row, col=col)
    for i, bound in enumerate([lo, hi]):
        fig.add_trace(go.Scatter(
            x=x_grid, y=bound, mode='lines', name=f'{name}: 95% conf. bounds', legendgroup=name,
            showlegend=show_legend and i == 0, line=dict(color=color, width=2, dash='dot'), hoverinfo='skip',
        ), row=row, col=col)
    return fit


def make_rate_vs_age_figure(df, rate, fit='lineage', title=None):
    # One rate (Sensitivity or Specificity) against embryo age, both marked lineages in the same plot.
    # fit: 'lineage' (one line per marked lineage), 'pooled' (one line over all embryos), 'both' or None.
    df = add_embryo_details(df)
    fig = go.Figure()
    fits = {}
    for lineage in LINEAGES:
        part = df[df['Lineage'] == lineage]
        if part.empty:
            continue
        fig.add_trace(go.Scatter(
            x=part['Age'], y=part[rate], mode='markers', name=f'{lineage} (n = {len(part)})', legendgroup=lineage,
            marker=dict(size=BIG_MARKER, color=LINEAGE_COLORS[lineage], line=dict(color='white', width=1)),
            hovertext=hover_texts(part), hovertemplate='%{hovertext}<extra></extra>',
        ))
        if fit in ('lineage', 'both') and len(part) >= 3:
            fits[lineage] = add_colored_fit(fig, part['Age'], part[rate], LINEAGE_COLORS[lineage], lineage, True,
                                            row=None, col=None, x_name='age')
    if fit in ('pooled', 'both'):
        fits['all'] = add_colored_fit(fig, df['Age'], df[rate], 'black', 'all embryos', True, row=None, col=None,
                                      x_name='age')

    style_axes(fig)
    fig.update_xaxes(range=list(AGE_LIM), title_text='age (# cells)', ticks='inside')
    fig.update_yaxes(title_text=rate.lower())
    fig.update_layout(
        title=dict(text=title if title is not None else f'{rate.lower()} vs age', x=0.01, xanchor='left'),
        template='simple_white', width=900, height=700, font=dict(family='Arial', size=BIG_FONT),
        hoverlabel=dict(align='left'),
        legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.18, yanchor='top'),
        margin=dict(l=90, r=30, t=80, b=200),
    )
    return fig, fits


def make_lineage_rates_figure(df, lineage, x_col='Age', width=900):
    # Sensitivity (left) and specificity (right) against x_col (age, or a distance column) for one marked lineage,
    # with the linear fit and its 95 % confidence bounds. Layout width : height = PDF_WIDTH_MM : PDF_HEIGHT_MM, so
    # save_figure_plotly(..., width_mm=PDF_WIDTH_MM) gives a page of that size (BIG_FONT = 18 px is then about 6.8 pt).
    part = add_embryo_details(df[df['Lineage'] == lineage])
    x = part[x_col]
    x_label = 'age (# cells)' if x_col == 'Age' else COLUMN_LABELS.get(x_col, x_col)
    pad = 0.05 * (x.max() - x.min())
    color = LINEAGE_COLORS[lineage]
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14)
    fits = {}
    for col, rate in [(1, 'Sensitivity'), (2, 'Specificity')]:
        fig.add_trace(go.Scatter(
            x=x, y=part[rate], mode='markers', name=f'{lineage} (n = {len(part)})',
            marker=dict(size=BIG_MARKER, color=color, line=dict(color='white', width=1)),
            hovertext=hover_texts(part), hovertemplate='%{hovertext}<extra></extra>',
        ), row=1, col=col)
        if len(part) >= 3:
            fits[rate] = add_colored_fit(fig, x, part[rate], color, lineage, False, row=1, col=col,
                                         x_name='age' if x_col == 'Age' else 'distance')
        fig.update_xaxes(range=[x.min() - pad, x.max() + pad], title_text=x_label, ticks='inside', row=1, col=col)
        fig.update_yaxes(title_text=rate.lower(), row=1, col=col)

    style_axes(fig)
    fig.update_layout(
        title=dict(text=f'{lineage} (n = {len(part)})', x=0.01, xanchor='left', y=0.98, yanchor='top'),
        template='simple_white', width=width, height=round(width * PDF_HEIGHT_MM / PDF_WIDTH_MM),
        font=dict(family='Arial', size=BIG_FONT), hoverlabel=dict(align='left'), showlegend=False,
        margin=dict(l=70, r=15, t=40, b=60),
    )
    return fig, fits


def save_table_figure(table, name, formats=None, font_size=12):
    # Draw a DataFrame as a table (bold header and first column, shaded alternate rows) and save it with save_figure
    # (PDF + PNG when SAVE_FIGURES). formats: {column: format spec}; integer columns default to 'd', others to '.3f'.
    formats = formats or {}
    shown = table.reset_index()
    text = pd.DataFrame({
        col: ['–' if pd.isna(v) else
              format(v, formats.get(col, 'd' if pd.api.types.is_integer_dtype(shown[col]) else '.3f'))
              for v in shown[col]]
        for col in shown.columns})

    fig, ax = plt.subplots(figsize=(1.6 * shown.shape[1], 0.4 * (len(shown) + 1)))
    ax.axis('off')
    tab = ax.table(cellText=text.to_numpy(), colLabels=list(shown.columns), cellLoc='right', colLoc='right',
                   loc='center')
    tab.auto_set_font_size(False)
    tab.set_fontsize(font_size)
    tab.auto_set_column_width(col=list(range(shown.shape[1])))
    tab.scale(1, 1.6)
    for (row, col), cell in tab.get_celld().items():
        if row == 0:
            cell.visible_edges = 'B'   # rule under the header only
            cell.set_linewidth(1.2)
        else:
            cell.set_linewidth(0)
            cell.set_facecolor('#f5f5f5' if row % 2 == 1 else 'white')
        if row == 0 or col == 0:
            cell.set_text_props(fontweight='bold')
    save_figure(fig, name)
    return fig

In [ ]:
def make_rates_vs_distance_figure(df, dist_col, fit='lineage'):
    # Sensitivity (left) and specificity (right) against the Procrustes distance, both marked lineages in each panel.
    # fit: 'lineage' (one line per marked lineage), 'pooled' (one line over all embryos), 'both' or None.
    df = add_embryo_details(df)
    x_label = COLUMN_LABELS.get(dist_col, dist_col)
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1)
    fits = {}
    for col, rate in [(1, 'Sensitivity'), (2, 'Specificity')]:
        first = col == 1   # legend entries from the left panel only
        for lineage in LINEAGES:
            part = df[df['Lineage'] == lineage]
            if part.empty:
                continue
            fig.add_trace(go.Scatter(
                x=part[dist_col], y=part[rate], mode='markers', name=f'{lineage} (n = {len(part)})',
                legendgroup=lineage, showlegend=first,
                marker=dict(size=BIG_MARKER, color=LINEAGE_COLORS[lineage], line=dict(color='white', width=1)),
                hovertext=hover_texts(part), hovertemplate='%{hovertext}<extra></extra>',
            ), row=1, col=col)
            if fit in ('lineage', 'both') and len(part) >= 3:
                fits[lineage, rate] = add_colored_fit(fig, part[dist_col], part[rate], LINEAGE_COLORS[lineage],
                                                      lineage, first, row=1, col=col)
        if fit in ('pooled', 'both'):
            fits['all', rate] = add_colored_fit(fig, df[dist_col], df[rate], 'black', 'all embryos', first,
                                                row=1, col=col)
        fig.update_xaxes(range=[0, df[dist_col].max() * 1.05], title_text=x_label, ticks='inside', row=1, col=col)
        fig.update_yaxes(title_text=rate.lower(), row=1, col=col)

    style_axes(fig)
    fig.update_layout(
        title=dict(text=f'sensitivity and specificity vs {x_label}', x=0.01, xanchor='left'),
        template='simple_white', width=1400, height=650, font=dict(family='Arial', size=BIG_FONT),
        hoverlabel=dict(align='left'),
        legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.2, yanchor='top'),
        margin=dict(l=90, r=30, t=80, b=150),
    )
    return fig, fits

## Figures

In [ ]:
method = 'Procrustes mod'
dist_col = DIST_COLUMNS[method]
df = results[method]

details = add_embryo_details(df)
color_range = (details[dist_col].min(), details[dist_col].max())

### Sensitivity and specificity per marked lineage, coloured by Procrustes distance

In [ ]:
fig = make_box_figure_by_distance(df, dist_col, color_range=color_range)
save_figure_plotly(fig, f'{method} boxes coloured by distance')
fig.show()

### Sensitivity and specificity vs age, both marked lineages

In [ ]:
fits_age = {}
for rate in ['Sensitivity', 'Specificity']:
    fig, fits_age[rate] = make_rate_vs_age_figure(df, rate, fit='lineage')
    save_figure_plotly(fig, f'{method} {rate} vs age both lineages')
    fig.show()
fit_table({(lineage, rate): f for rate, fits in fits_age.items() for lineage, f in fits.items()})

### Sensitivity and specificity vs age, per marked lineage (PDF 120 mm × 44 mm)

In [ ]:
for lineage in LINEAGES:
    if (df['Lineage'] == lineage).sum() < 3:
        print(f'{lineage}: fewer than 3 embryos, skipped')
        continue
    fig, fits_lineage = make_lineage_rates_figure(df, lineage)
    save_figure_plotly(fig, f'{method} {lineage} rates vs age', width_mm=PDF_WIDTH_MM)
    fig.show()
    display(fit_table(fits_lineage))

### Excluding embryos by Procrustes distance

Mean and median sensitivity and specificity of the embryos with a Procrustes distance at or below each threshold
(`all` row: both marked lineages). At the largest threshold no embryo is excluded.

In [ ]:
thresholds = [0.015, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.12]
sweep = pd.concat({t: accuracy_by_distance(details, t, exclude='above', dist_col=dist_col).loc['all']
                   for t in thresholds}, axis=1).T
sweep = sweep.astype({'embryos kept': int, 'excluded': int})
sweep.index.name = 'threshold'

fig = save_table_figure(sweep, f'{method} threshold sweep')
plt.show()
sweep.round(3)

### Sensitivity and specificity versus Procrustes distance.

In [ ]:
fig, fits_dist = make_rates_vs_distance_figure(df, dist_col, fit='lineage')
save_figure_plotly(fig, f'{method} rates vs distance both lineages')
fig.show()
fit_table(fits_dist).rename(columns={'slope (per cell)': 'slope (per unit distance)'})


In [ ]:
for lineage in LINEAGES:
    if (df['Lineage'] == lineage).sum() < 3:
        print(f'{lineage}: fewer than 3 embryos, skipped')
        continue
    fig, fits_lineage = make_lineage_rates_figure(df, lineage, x_col=dist_col)
    save_figure_plotly(fig, f'{method} {lineage} rates vs distance', width_mm=PDF_WIDTH_MM)
    fig.show()
    display(fit_table(fits_lineage).rename(columns={'slope (per cell)': 'slope (per unit distance)'}))
